## CS5242 Neural Networks and Deep Learning
## Sem 2 2025/26
### Lecturer: Xavier Bresson
### Teaching Assistants: Yuwei Zeng, KinWhye Chew, Joel Loo Enming, Ryoji Kubo

## Midterm exam, coding test
Date: Mar 2 2026 <br>
Time: 6:30pm-8:00pm (90min) <br>

*Instructions* <br>
Name: Please, add your name here : <br>
Answers: Please write your answers directly in this notebook by completing the code sections marked with  
`# YOUR CODE STARTS HERE`  
`# YOUR CODE` (it can span one or multiple lines)  
`# YOUR CODE ENDS HERE`. <br>
Remark: Ensure your code runs without errors. No points will be awarded for buggy or incomplete code.  
Remark: If certain conditions of the questions (for eg. hyperparameter values) are not stated, you are free to choose anything you want.

## Exercise 1 : Implement a 2D Convolution Operation

In this exercise, you will implement a **2D valid convolution** over a single-channel 2D image from scratch using basic tensor operations.

For an input image $X \in \mathbb{R}^{H \times W}$ and a kernel filter $W \in \mathbb{R}^{K_H \times K_W}$, the valid convolution output $Y$ computes:
$$
Y_{i, j} = \sum_{u=0}^{K_H-1} \sum_{v=0}^{K_W-1} X_{i+u, j+v} W_{u, v}
$$
where the output $Y$ has dimension $(H - K_H + 1) \times (W - K_W + 1)$.

**[Your Tasks]**

Please complete the following item:
1) The `conv2d_manual()` function that computes the valid 2D convolution between `image` and `kernel`. You should use nested loops over the output spatial dimensions and compute the element-wise sum of products.

**[Implementation Criteria]**

You will be running a small convolution on a $5 \times 5$ input image with a $3 \times 3$ kernel. If implemented correctly, your output will match the expected tensor and the script will print **"Well Done!"**.

**[Useful Functions]**

- `torch.zeros(h, w)` Creates a tensor of size $h \times w$ filled with zeros.
- `torch.sum(tensor)` Returns the sum of all elements in the `tensor`.

In [ ]:
import torch

# 2D Matrix Multiplication
mat1 = torch.tensor([[1, 2], [3, 4]])
mat2 = torch.tensor([[5, 6], [7, 8]])

# Using @ operator
result_at = mat1 @ mat2
result_at
print(mat1.shape)


# All yield: tensor([[19, 22], [43, 50]])


In [ ]:
%reset -f
import datetime
import torch
print('Timestamp:',datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S"))

def conv2d_manual(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape
    out_h = h - kh + 1
    out_w = w - kw + 1
    out = torch.zeros((out_h, out_w))
    
    #############################################
    # YOUR CODE STARTS HERE
    for i in range(out_h):
        for j in range(out_w):
            out[i, j] = torch.sum(image[i:i+kh, j:j+kw] * kernel)
    # YOUR CODE ENDS HERE
    #############################################
    
    return out

image = torch.tensor([
    [1.0, 2.0, 3.0, 0.0, 1.0],
    [0.0, 1.0, 2.0, 1.0, 0.0],
    [2.0, 1.0, 0.0, 1.0, 2.0],
    [1.0, 0.0, 1.0, 2.0, 1.0],
    [0.0, 2.0, 1.0, 0.0, 1.0]
])

kernel = torch.tensor([
    [1.0, 0.0, -1.0],
    [1.0, 0.0, -1.0],
    [1.0, 0.0, -1.0]
])

out = conv2d_manual(image, kernel)
expected_out = torch.tensor([
    [-2.,  2.,  2.],
    [ 0., -2.,  0.],
    [ 1.,  0., -2.]
])

if out.shape == (3, 3) and (out - expected_out).abs().sum() < 1e-4:
    print("Well Done!")
else:
    print("Try again.")

## Exercise 2 : Implement Max Pooling 2D

In this exercise, you will implement the **Max Pooling 2D** operation which is commonly found in Convolutional Neural Networks to downsample spatial dimensions.

Given an input image $X \in \mathbb{R}^{H \times W}$, a max pooling patch size of $K \times K$ with a matching stride $K$ reduces the spatial size by taking the maximum value over each disjoint $K \times K$ block:
$$
Y_{i, j} = \max_{0 \leq u < K, 0 \leq v < K} X_{i \cdot K + u, j \cdot K + v}
$$
The output $Y$ has dimension $(H // K) \times (W // K)$.

**[Your Tasks]**

Please complete the following item:
1) The `maxpool2d_manual()` function that computes the max pooling over the input `image`. Use nested loops over the output dimensions and take the maximum inside the pooling window.

**[Implementation Criteria]**

You will run max pooling on an $8 \times 8$ image with a pool size $K=2$. If correct, the script will print **"Well Done!"**.

**[Useful Functions]**

- `torch.zeros(h, w)` Creates a tensor of size $h \times w$ filled with zeros.
- `torch.max(tensor)` Returns the maximum value of all elements in the `tensor`.

In [ ]:
8 // 2

In [ ]:
%reset -f
import datetime
import torch
print('Timestamp:',datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S"))

def maxpool2d_manual(image, K):
    h, w = image.shape
    out_h = h // K
    out_w = w // K
    out = torch.zeros((out_h, out_w))
    
    #############################################
    # YOUR CODE STARTS HERE
    for i in range(out_h):
        for j in range(out_w):
            out[i, j] = torch.max(image[i*K:(i+1)*K, j*K:(j+1)*K])
    # YOUR CODE ENDS HERE
    #############################################
    
    return out

image = torch.arange(64, dtype=torch.float32).view(8, 8)
image[1, 1], image[0, 5], image[5, 4] = 99.0, 100.0, 88.0

out = maxpool2d_manual(image, 2)
expected_out = torch.tensor([
    [ 99.,  11., 100.,  15.],
    [ 25.,  27.,  29.,  31.],
    [ 41.,  43.,  88.,  47.],
    [ 57.,  59.,  61.,  63.]
])

if out.shape == (4, 4) and (out - expected_out).abs().sum() < 1e-4:
    print("Well Done!")
else:
    print("Try again.")

## Exercise 3 : Implement Scaled Dot-Product Attention

In this exercise, you will manually compute the core mechanism of the Transformer architecture: **Scaled Dot-Product Attention**.

Given Query $Q$, Key $K$, and Value $V$ matrices of shape $N \times d_k$, where $N$ is the sequence length and $d_k$ is the embedding dimension, the attention output is computed as:
$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V
$$
where the softmax function is applied over the last dimension.

**[Your Tasks]**

Please complete the following items:
1) Calculate the raw attention scores $\frac{Q K^T}{\sqrt{d_k}}$.
2) Apply the Softmax activation to obtain the attention weights.
3) Multiply the weights with $V$ to get the final context vectors.

**[Implementation Criteria]**

The script evaluates your function with randomly generated $Q, K, V$ matrices. If the output and attention weights match the expected calculations, it will print **"Well Done!"**.

**[Useful Functions]**

- `torch.matmul(A, B)` Performs matrix multiplication between A and B.
- `A.transpose(dim0, dim1)` Returns a tensor that is a transposed version of `A` with dimensions swapped.
- `torch.sqrt(tensor)` Returns a new tensor with the square-root of the elements of `tensor`.
- `torch.softmax(tensor, dim)` Applies the Softmax function over a specific dimension `dim`.

In [ ]:
%reset -f
import datetime
import torch
import math
print('Timestamp:',datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S"))

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.size(-1)
    
    #############################################
    # YOUR CODE STARTS HERE
    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    # YOUR CODE ENDS HERE
    #############################################
    
    return output, attn_weights

torch.manual_seed(42)
Q = torch.randn(4, 8)
K = torch.randn(4, 8)
V = torch.randn(4, 8)

out, attn = scaled_dot_product_attention(Q, K, V)

expected_attn = torch.tensor([
    [0.1942, 0.4102, 0.3242, 0.0714],
    [0.0408, 0.6555, 0.0864, 0.2173],
    [0.1041, 0.2057, 0.1824, 0.5078],
    [0.1926, 0.1803, 0.1955, 0.4316]
])

if attn.shape == (4, 4) and out.shape == (4, 8) and (attn - expected_attn).abs().sum() < 1e-2:
    print("Well Done!")
else:
    print("Try again.")

## Exercise 4 : Construct a Multi-Head Attention Forward Pass

In this exercise, you will expand the scaled dot-product attention into a **Multi-Head Attention** module. Instead of performing a single attention function, this mechanism linearly projects the queries, keys, and values $h$ times ($h$ is the number of heads) utilizing learned weights, applies attention in parallel, concatenates the head outputs, and performs a final linear projection.

For a single input sequence $X \in \mathbb{R}^{N \times d_{\text{model}}}$, we first compute queries, keys, and values for all heads combined:
$$
Q = X W_Q, \quad K = X W_K, \quad V = X W_V
$$
where $W_Q, W_K, W_V \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$.
We then reshape these vectors to split the $d_{\text{model}}$ dimensions into $h$ heads of dimension $d_k = d_{\text{model}} / h$. The attention is then computed over each head, the transposed results are concatenated, and finally a linear map $W_O \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$ is applied.

**[Your Tasks]**

Complete the missing implementations in `__init__` and `forward`:
1) **Linear Layers:** Initialize `W_q`, `W_k`, `W_v`, and `W_o` as `nn.Linear` layers with `bias=False`.
2) **Projection & Reshaping:** Within `forward()`, compute $Q, K, V$. Reshape them to incorporate the `num_heads` dimension and transpose them so that the shapes are `(N, num_heads, d_k)` and then `(num_heads, N, d_k)` or similar to perform attention broadcasting.
3) **Attention Calculation:** Calculate the attention context over the $h$ heads.
4) **Concatenation & Output:** Transpose and reshape the context matrices back to `(N, d_model)` and pass it through $W_O$. 

**[Implementation Criteria]**

We will pass an input sequence $X$ through the instantiated module and verify whether the output shape and internal calculations match the reference outputs. 

**[Useful Functions]**

- `nn.Linear(in_features, out_features, bias)`
- `tensor.view(*shape)` Reshapes a tensor.
- `tensor.transpose(dim0, dim1)` Swaps two dimensions.
- `tensor.contiguous()` Ensures the memory tensor is contiguous before `view()` is called.

In [ ]:
%reset -f
import datetime
import torch
import torch.nn as nn
print('Timestamp:',datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S"))

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        
        #############################################
        # YOUR CODE STARTS HERE
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        # YOUR CODE ENDS HERE
        #############################################
        
    def forward(self, x):
        N, _ = x.shape
        
        #############################################
        # YOUR CODE STARTS HERE
        Q = self.W_q(x).view(N, self.num_heads, self.d_k).transpose(0, 1)
        K = self.W_k(x).view(N, self.num_heads, self.d_k).transpose(0, 1)
        V = self.W_v(x).view(N, self.num_heads, self.d_k).transpose(0, 1)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        
        context = context.transpose(0, 1).contiguous().view(N, self.d_model)
        output = self.W_o(context)
        # YOUR CODE ENDS HERE
        #############################################
        
        return output

# Set a manual seed and initialize model
torch.manual_seed(42)
d_model = 16
num_heads = 4
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# To ensure determinism, let's manually re-initialize the weights to small values
with torch.no_grad():
    for name, param in mha.named_parameters():
        param.copy_(torch.randn(param.shape) * 0.1)

# Generate dummy sequence data
X = torch.randn(5, 16)
output = mha(X)

expected_sum = torch.tensor(-0.6871)

if output.shape == (5, 16) and (output.sum() - expected_sum).abs() < 1e-3:
    print("Well Done!")
else:
    print("Try again.")

## End of coding test